# 03 Attribute heads on frozen CLIP embeddings

**Compute bucket: Colab/Kaggle (a GPU makes the embedding pass quick; CPU works, just slower).**

Trains two small MLP heads on top of frozen CLIP ViT-B/32 embeddings of object crops:

* **category** (18 classes) from the labelled boxes in the dataset from notebook `01`
* **material** (optional): put a CSV at `MyDrive/reunite/material.csv` with `image,material` (plastic, metal, leather, fabric, glass, paper, wood, rubber); skipped if absent

It also measures the zero-shot baseline on the same validation crops, so the gain over "no training" is a number you can put on a slide.
Output: `heads.pt` in the exact format `backend/app/ml/vision/attributes.py` reads.

In [ ]:
!pip -q install open_clip_torch pillow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, zipfile, random
from pathlib import Path
import numpy as np, torch, open_clip
from PIL import Image

CLASSES = [
    "id_card", "laptop", "phone", "tablet", "earbuds", "headphones", "charger", "power_bank", "wallet",
    "keys", "backpack", "bottle", "book", "notebook", "spectacles", "watch", "umbrella", "calculator",
]  # same order and names as backend/app/ml/text/lexicons/categories.yaml
MATERIALS = ["plastic", "metal", "leather", "fabric", "glass", "paper", "wood", "rubber"]
DEST = Path("/content/drive/MyDrive/reunite")
DATA = Path("/content/dataset")
if not DATA.exists():
    with zipfile.ZipFile(DEST / "reunite_dataset.zip") as z:
        z.extractall(DATA)
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model = model.to(device).eval()
tok = open_clip.get_tokenizer("ViT-B-32")
random.seed(0); torch.manual_seed(0)

In [ ]:
def crops(split, limit=None):
    items = []
    for lab in sorted((DATA / "labels" / split).glob("*.txt")):
        img = DATA / "images" / split / f"{lab.stem}.jpg"
        for line in lab.read_text().splitlines():
            c, x, y, w, h = line.split(); items.append((img, int(c), float(x), float(y), float(w), float(h)))
    random.shuffle(items)
    return items[:limit] if limit else items

@torch.no_grad()
def embed(items, batch=128):
    embs, ys = [], []
    for i in range(0, len(items), batch):
        chunk = items[i:i + batch]; tensors = []
        for img, c, x, y, w, h in chunk:
            im = Image.open(img).convert("RGB"); W, H = im.size
            box = (max((x - w / 2) * W, 0), max((y - h / 2) * H, 0), min((x + w / 2) * W, W), min((y + h / 2) * H, H))
            crop = im.crop(tuple(int(v) for v in box)) if box[2] - box[0] > 24 and box[3] - box[1] > 24 else im
            tensors.append(preprocess(crop))
        e = model.encode_image(torch.stack(tensors).to(device)).float()
        embs.append(torch.nn.functional.normalize(e, dim=-1).cpu()); ys += [c for _, c, *_ in chunk]
    return torch.cat(embs), torch.tensor(ys)

Xtr, ytr = embed(crops("train")); Xva, yva = embed(crops("val"))
print(Xtr.shape, Xva.shape)

In [ ]:
# zero-shot baseline on the same validation crops (what the backend does without heads.pt)
PHRASE = {"id_card": "student ID card", "power_bank": "power bank", "earbuds": "pair of wireless earbuds", "headphones": "pair of headphones",
          "keys": "bunch of keys", "spectacles": "pair of eyeglasses", "bottle": "water bottle", "charger": "phone charger", "notebook": "paper notebook", "watch": "wrist watch"}
TEMPLATES = ["a photo of a {}.", "a photo of a lost {}.", "a {} lying on a table.", "a close-up photo of a {}."]
with torch.no_grad():
    rows = []
    for c in CLASSES:
        p = PHRASE.get(c, c.replace("_", " "))
        e = torch.nn.functional.normalize(model.encode_text(tok([t.format(p) for t in TEMPLATES]).to(device)).float(), dim=-1).mean(0)
        rows.append(torch.nn.functional.normalize(e, dim=-1).cpu())
    T = torch.stack(rows)
zs_acc = ((100 * Xva @ T.T).argmax(1) == yva).float().mean().item()
print(f"zero-shot category accuracy: {zs_acc:.3f}")

In [ ]:
def train_head(Xtr, ytr, Xva, yva, n_classes, hidden=256, epochs=60, lr=1e-3, wd=1e-4):
    net = torch.nn.Sequential(torch.nn.Linear(Xtr.shape[1], hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, n_classes))  # keys 0.* and 2.* (attributes.py rebuilds this)
    counts = torch.bincount(ytr, minlength=n_classes).float().clamp(min=1)
    w = (counts.sum() / counts) / n_classes                               # class-balanced loss
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    best, best_state, bad = 0.0, None, 0
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), 256):
            idx = perm[i:i + 256]
            loss = torch.nn.functional.cross_entropy(net(Xtr[idx]), ytr[idx], weight=w, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            acc = (net(Xva).argmax(1) == yva).float().mean().item()
        if acc > best: best, best_state, bad = acc, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 10: break
    net.load_state_dict(best_state)
    return net, best

cat_net, cat_acc = train_head(Xtr, ytr, Xva, yva, len(CLASSES))
print(f"trained category head accuracy: {cat_acc:.3f}  (zero-shot {zs_acc:.3f})")

In [ ]:
# optional material head from a CSV of weak labels
mat_blob, mat_acc = None, None
csv_path = DEST / "material.csv"
if csv_path.exists():
    import csv
    rows = [r for r in csv.DictReader(open(csv_path)) if r["material"] in MATERIALS]
    random.shuffle(rows)
    embs, ys = [], []
    with torch.no_grad():
        for r in rows:
            e = model.encode_image(preprocess(Image.open(r["image"]).convert("RGB")).unsqueeze(0).to(device)).float()
            embs.append(torch.nn.functional.normalize(e, dim=-1).cpu()[0]); ys.append(MATERIALS.index(r["material"]))
    X, y = torch.stack(embs), torch.tensor(ys); cut = int(len(X) * 0.85)
    mat_net, mat_acc = train_head(X[:cut], y[:cut], X[cut:], y[cut:], len(MATERIALS), hidden=128)
    mat_blob = {"in_dim": 512, "hidden": 128, "classes": MATERIALS, "state_dict": {k: v.cpu() for k, v in mat_net.state_dict().items()}}
    print(f"material head accuracy: {mat_acc:.3f}")
else:
    print("no material.csv: material stays zero-shot")

In [ ]:
blob = {"category": {"in_dim": 512, "hidden": 256, "classes": CLASSES, "state_dict": {k: v.cpu() for k, v in cat_net.state_dict().items()}}}
if mat_blob: blob["material"] = mat_blob
torch.save(blob, DEST / "heads.pt")
json.dump({"category_accuracy": cat_acc, "zero_shot_category_accuracy": zs_acc, "material_accuracy": mat_acc, "n_train": len(Xtr), "n_val": len(Xva), "classes": CLASSES},
          open(DEST / "heads_metrics.json", "w"), indent=2)
print("saved heads.pt and heads_metrics.json to", DEST)